#### Starting With a Basic Form of Self-Attention

Before we normalize the $\omega_{ij}$ values to obtain the attention weights, $a_{ij}$, let's illustrate how we compute the $\omega_{ij}$ values with a code example. Here, let's assume we have an input sentence "can you help me to translate this sentence" that has already been mapped to an integer representation via a dictionary:

In [1]:
import torch
sentence = torch.tensor(
    [0, # can
     7, # you
     1, # help
     2, # me
     5, # to
     6, # translate
     4, # this
     3] # sentence
)

sentence

tensor([0, 7, 1, 2, 5, 6, 4, 3])

Let's also assume that we already encoded this sentence into a real-number vector representation via an embedding layer. Here, our embedding size is 16, and we assume that the dictionary size is 10. The following code will produce the word embeddings of our eight words:

In [2]:
torch.manual_seed(123)
embed = torch.nn.Embedding(10, 16)
embedded_sentence = embed(sentence).detach()
embedded_sentence.shape

torch.Size([8, 16])

Now, we can compute $\omega_{ij}$ as the dot product between the ith and jth word embeddings. We can do this for all $\omega_{ij}$ values as follows:

In [3]:
omega = torch.empty(8, 8)
for i, x_i in enumerate(embedded_sentence):
    for j, x_j in enumerate(embedded_sentence):
        omega[i, j] = torch.dot(x_i, x_j)

While the preceding code is easy to read and understand, for loops can be very inefficient, so let's compute this using matrix multiplication instead:

In [4]:
omega_mat = embedded_sentence.matmul(embedded_sentence.T)

We can use the torch.allclose function to check that this matrix multiplication produces the expected results. If two tensors contain the same values, torch.allclose returns True, as we can see here:

In [5]:
torch.allclose(omega, omega_mat)

True

We can compute the attention weights using PyTorch's softmax function as follows:

In [6]:
import torch.nn.functional as F
attention_weights = F.softmax(omega, dim=1)
attention_weights.shape

torch.Size([8, 8])

Note that the attention_weights is an $8 \times 8$ matrix, where each element represents an attention weight, $\alpha_{ij}$. For instance, if we are processing the $ith$ input word, the $ith$ row of this matrix contains the corresponding attention weights for all words in the sentence. These attention weights indicate how relevant each word is to the $ith$ word. Hence, the columns in this attention matrix should sum to 1, which we can confirm via the following code:

In [7]:
attention_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

Lastly, let us see a code example for computing the context vectors, $z^{(i)}$, as the attention-weighted sum of the inputs. In particular, let's assume we are computing the context vector for the second input word, that is, $z^{(2)}$:

In [8]:
x_2 = embedded_sentence[1, :]
context_vec_2 = torch.zeros(x_2.shape)
for j in range(8):
    x_j = embedded_sentence[j, :]
    context_vec_2 += attention_weights[1, j] * x_j
context_vec_2

tensor([-9.3975e-01, -4.6856e-01,  1.0311e+00, -2.8192e-01,  4.9373e-01,
        -1.2896e-02, -2.7327e-01, -7.6358e-01,  1.3958e+00, -9.9543e-01,
        -7.1287e-04,  1.2449e+00, -7.8077e-02,  1.2765e+00, -1.4589e+00,
        -2.1601e+00])

Again, we can achieve this more efficiently by using matrix multiplication. Using the following code, we are computign the context vectors for all eight input words:

In [9]:
context_vectors = torch.matmul(attention_weights, embedded_sentence)

Similarly to the input word embeddings stored in embedded_sentence, the context_vectors matrix has dimensionality $8 \times 16$. The second row in this matrix contains the context vector for the second input word, and we can check the implementation using torch.allclose() again:

In [10]:
torch.allclose(context_vec_2, context_vectors[1])

True

As we can see, the manual for loop and matrix multiplication of the second context vector yielded the same results.

#### Parametrizing the Self-Attention Mechanism: Scaled Dot-Product Attention

We can initialize the projectin matrices as follows:

In [11]:
torch.manual_seed(123)
d = embedded_sentence.shape[1]
U_query = torch.rand(d, d)
U_key = torch.rand(d, d)
U_value = torch.rand(d, d)

Using the query projection matrix, we can then compute the query sequence. For this example, consider the second input element, $x^{(2)}$, as our query:

In [12]:
x_2 = embedded_sentence[1]
query_2 = U_query.matmul(x_2)

In a similar fashion, we can compute the key and value sequences, $k^{(i)}$ and $v^{(i)}$:

In [13]:
key_2 = U_key.matmul(x_2)
value_2 = U_value.matmul(x_2)

However, as we can see, we also need the key and value sequences for all other input elements, which we can compute as follows:

In [14]:
keys = U_key.matmul(embedded_sentence.T).T
values = U_value.matmul(embedded_sentence.T).T

In the key matrix, the $ith$ row corresponds to the key sequence of the $ith$ input element, and the same applies to the value matrix. We can confirm this by using torch.allclose() again, which should return True:

In [15]:
print(torch.allclose(key_2, keys[1]))
print(torch.allclose(value_2, values[1]))

True
True


In the previous section, we computed the unnormalized weights, $\omega_{ij}$, as the pairwise dot product between the given input sequence element, $x^{(i)}$, and the $jth$ sequence element, $x^{(j)}$. Now, in this parametrized version of self-attention, we compute $\omega_{ij}$ as the dot product between the query and the key:

$$
\omega_{ij} = q^{(i)^{T}}k^{(j)}
$$

For example, the following code computes the unnormalized attention weight, $\omega_{23}$, that is, the dot product between our query and the third input sequence element:

In [16]:
omega_23 = query_2.dot(keys[2])
omega_23

tensor(14.3667)

Since we will be needing these later, we can scale up this computation to all keys:

In [17]:
omega_2 = query_2.matmul(keys.T)
omega_2

tensor([-25.1623,   9.3602,  14.3667,  32.1482,  53.8976,  46.6626,  -1.2131,
        -32.9392])

The next step in self-attention is to go from the unnormalized attention weights, $w_{ij}$, to the normalized attention weights, $\alpha_{ij}$, using the softmax function. We can then further use $1/\sqrt{m}$ to scale $w_{ij}$ before normalizing it via the softmax function, as follows:

$$
\alpha_{ij} = softmax\Bigg(\frac{\omega_{ij}}{\sqrt{m}}\Bigg)
$$

Note that scaling $\omega_{ij}$ by $1/\sqrt{m}$, where typically $m = d_k$, ensures the Euclidean length of the weight vectors will be approximately in the same range.

The following code for implementing this normalization to compute the attention weights for the entire input sequence with respect to the second input element as the query:

In [18]:
attention_weights_2 = F.softmax(omega_2 / d**0.5, dim=0)
attention_weights_2

tensor([2.2317e-09, 1.2499e-05, 4.3696e-05, 3.7242e-03, 8.5596e-01, 1.4026e-01,
        8.8897e-07, 3.1935e-10])

Finally, the output is a weighted average of value sequences $z^{(i)} = \sum_{j=1}^{T}\alpha_{ij}v^{(j)}$, which can be implemented as follows:

In [19]:
context_vector_2 = attention_weights_2.matmul(values)
context_vector_2

tensor([-1.2226, -3.4387, -4.3928, -5.2125, -1.1249, -3.3041, -1.4316, -3.2765,
        -2.5114, -2.6105, -1.5793, -2.8433, -2.4142, -0.3998, -1.9917, -3.3499])

#### Encoding Context Embeddings Via Multi-Head Attention

To illustrate the multi-head self-attention stack in code, first consider how we created the single query projection matrix in the previous subsection, Parametrizing the Self-Attention Mechanism: Scaled Dot-Product Attention:

In [20]:
torch.manual_seed(123)
d = embedded_sentence.shape[1]
one_U_query = torch.rand(d, d)

Now, assume we have eight attention heads similar to the original transformer, that is, $h=8$

In [21]:
h = 8
multihead_U_query = torch.rand(h, d, d)
multihead_U_key = torch.rand(h, d, d)
multihead_U_value = torch.rand(h, d, d)

As we can see in the code, multiple attention heads can be added by simply adding an additional dimension.

In practice, rather than having a separate matrix for each attention head, transformer implementations use a single matrix for all attention heads. The attention heads are then organized into logically  separate regions in this matrix, which can be accessed via Boolean masks. This makes it possible to implement multi-head attention more efficiently because multiple matrix multiplications can be implemented as a single matrix multiplication instead. However, for simpliciy, we are omitting this implementation detail in this section.

After initializing the projection matrices, we can compute the projected sequences similar to how it's done in scaled dot-product attention. Now, instead of computing one set of query, key, and value sequences, we need to compute $h$ sets of them. More formally, for example, the computation involving the query projection for the $ith$ data point in the $jth$ head can be written as follows:

$$
q_j^{(i)} = U_{q_{j}}x^{(i)}
$$

We then repeat this combination for all heads $j \in \{1, \ldots, h\}$

In code, this looks like the following for the second input word as the query:

In [22]:
multihead_query_2 = multihead_U_query.matmul(x_2)
multihead_query_2.shape

torch.Size([8, 16])

Similarly, we can compute key and value sequences for each head:

In [23]:
multihead_key_2 = multihead_U_key.matmul(x_2)
multihead_value_2 = multihead_U_value.matmul(x_2)
multihead_key_2[2]

tensor([-1.9619, -0.7701, -0.7280, -1.6840, -1.0801, -1.6778,  0.6763,  0.6547,
         1.4445, -2.7016, -1.1364, -1.1204, -2.4430, -0.5982, -0.8292, -1.4401])

The code output shows the key vector of the second input element via the third attention head.

However, remember that we need to repeat the key and value computations for all input sequence elemnts, not just x_2 - we need this to compute self-attention later. A simple and illustrative  way to do this is by expanding the input sequence embeddings to size 8 as the first dimension, which is the number of attention heads. We use the .repeat() method for this:

In [24]:
stacked_inputs = embedded_sentence.T.repeat(8, 1, 1)
stacked_inputs.shape

torch.Size([8, 16, 8])

Then, we can have a batch matrix multiplication, via torch.bmm(), with the attention heads to compute all keys:

In [25]:
multihead_keys = torch.bmm(multihead_U_key, stacked_inputs)
multihead_keys.shape

torch.Size([8, 16, 8])

In this code, we now have a tensor that refers to the eight attention heads in its first dimension. The second and third dimensions refer to the embedding size and the number of words, respectively. Let us swap the second and third dimensions so that the keys have a more intuitive representation, that is, the same dimensionality as the original input sequence embedded_sentence:

In [26]:
multihead_keys = multihead_keys.permute(0, 2, 1)
multihead_keys.shape

torch.Size([8, 8, 16])

After rearranging, we can access the second key value in the second attention head as follows:

In [27]:
multihead_keys[2, 1]

tensor([-1.9619, -0.7701, -0.7280, -1.6840, -1.0801, -1.6778,  0.6763,  0.6547,
         1.4445, -2.7016, -1.1364, -1.1204, -2.4430, -0.5982, -0.8292, -1.4401])

We can see that this is the same key value that we got via multihead_key_2[2] earlier, which indicates that our complex matrix manipulations and computations are correct. So let's repeat it for the value sequences:

In [28]:
multihead_values = torch.matmul(multihead_U_value, stacked_inputs)
multihead_values = multihead_values.permute(0, 2, 1)

We follow the steps of the single head attention calculation to calculate the context vectors as described in the Parameterizing the Self-Attention Mechanism: Scaled-Dot Product Attention section. We will skip the intermediate steps for brevity and assume that we have computed the context vectors for the second input element as the query and the eight different attention heads, which we represent as multihead_z_2 via random data:

In [29]:
multihead_z_2 = torch.rand(8, 16)

Note that the first dimension indexes over the eight attention heads, and the context vectors, similar to the input sequences, are 16-dimensional vectors. If this appears complicated, think of multihead_z_2 as eight copies of the $z^{(2)}$, that is, we have one for each of the eight attention heads.

Then, we concatenate these vectors into one long vector of length $d_v \times h$ and use a linear projection (via a fully connected layer) to map it back to a vector of length $d_v$.

In code, we can implement the concatenation and squashing as follows:

In [30]:
linear = torch.nn.Linear(8*16, 16)
context_vector_2 = linear(multihead_z_2.flatten())
context_vector_2.shape

torch.Size([16])

#### Using GPT-2 to Generate New Text

We will be accessing GPT-2 via transformers, which is a very comprehensive Python library created by Hugging Face that provides various transformer-based models for pre-training and fine-tuning.

Once we have installed the transformers library, we can run the following code to import a pre-trained GPT model that can generate new text:

In [31]:
from transformers import GenerationConfig, pipeline, set_seed
generator = pipeline("text-generation", model="gpt2")
config = GenerationConfig(max_new_tokens=15, do_sample=True,
                          temperature=0.7, num_return_sequences=3)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [32]:
set_seed(123)
generator('Hey readers, today is',
          generation_config=config,
          clean_up_tokenization_spaces=False)

[{'generated_text': "Hey readers, today is the day the movie comes out. I'm excited that this week's release"},
 {'generated_text': 'Hey readers, today is our day off, so I wanna share some of the highlights of your life'},
 {'generated_text': 'Hey readers, today is the day to get your hands on some of these awesome books from the award'}]

As we can see from the output, the model generated three reasonable sentences based on our text snippet.

Also, we can use a transformer model to generate features for training other models. The following code illustrates how we can use GPT-2 to generate features based on an input text:

In [33]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
text = "Let us encode this sentence"
encoded_input = tokenizer(text, return_tensors='pt')
encoded_input

{'input_ids': tensor([[ 5756,   514, 37773,   428,  6827]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

This code encoded the input sentence text into a tokenized format for the GPT-2 model. As we can see, it mapped the strings to an integer representation, and it set the attention ask to all 1s, which means that all words will be processed when we pass the encoded input to the model, as shown here:

In [34]:
from transformers import GPT2Model
model = GPT2Model.from_pretrained("gpt2")
output = model(**encoded_input)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The output variable stores the last hidden state, that is, our GPT-2 based feature encoding of the input sentence:

In [35]:
output['last_hidden_state'].shape

torch.Size([1, 5, 768])

To suppress the verbose output, we only showed the shape of the tensor. Its first dimension is the batch size (we only have one input text), which is followed by the sentence length and size fo the feature encoding here. Here, each of the five words is encoded as a 768-dimnensional vector.

Now, we could apply this feature encoding to a given dataset and train a downstream classifier based on the GPT-2 based feature representation instead of using a bag-of-words model as discussed previously.

Moreover, an alternative approach to using large pre-trained language models is fine-tuning, as we discussed earlier.

#### Fine-Tuning a BERT Model in PyTorch

Now that we have introduced and discussed all the necessary concepts and the theory behind the original transformer and popular transformer-based models, it's time to take a look at the more preactical part. In this section, we will learn how to fine-tune a BERT model for **sentiment classification** in PyTorch.

Note that although there are many other transformer-baesd models to choose from, BERT provides a nice balance between model popularity and having a manageable model size so that it can be fine-tuned on a single GPU. Note also that pre-training a BERT from scratch is painful and quite unnecessary considering the availability of the transformers Python package provided by Hugging Face, which includes a bunch of pre-trained models that are ready for fine-tuning.

In the following sections, we'll see how to prepare and tokenize the IMDb movie review dataset and fine-tune the distilled BERT model to perform sentiment classification. We deliberately chose sentiment clsasification as a simple but classic example, though there are many other fascinating applications of language models. Also, by using the familiar IMDb movie review dataset, we can get a good idea of the predictive performance of the BERT model by comparing it to the logistic regression model in the previous chapters.

##### Loading the IMDb Movie Review Dataset

In this subsection, we will begin by loading the required packages and the dataset, split into train, validation, and test sets.

The **DistilBERT** model we are using in this chapter is a lightweight transformer model created by distilling a pre-trained BERT base model. The original uncased BERT base model contains over 110 million parameters while DistillBERT has 40 percent fewer parameters. Also, DistilBERT runs 60 percent faster and still preserves 95 percent of BERT's performance on the GLUE language understanding benchmark.

The following code imports all the packages we will be using in this chapter to prepare the data and fine-tune the DistilBERT model:

In [36]:
import time

import pandas as pd
import requests
import torch
import torch.nn.functional as F

import transformers
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification


Next, we specify some general settings, including the number of epochs we train the network on, the device specification, and the random seed. To reproduce the results, make sure to set a specific random seed such as 123:

In [37]:
torch.backends.cudnn.deterministic = True
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 3

We will be working on the IMDb movie review dataset, which we have already seen in Chapters 8 and 15. We load the data into a pandas DataFrame and make sure it looks all right:

In [38]:
df = pd.read_csv("movie_data.csv")
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


The next step is to split the dataset into separate training, validation, and test sets. Here, we use 70 percent of the reviews for the training set, 10 percent for the validation set, and the remaining 20 percent for testing:

In [39]:
train_texts = df.iloc[:35000]['review'].values
train_labels = df.iloc[:35000]['sentiment'].values
valid_texts = df.iloc[35000:40000]['review'].values
valid_labels = df.iloc[35000:40000]['sentiment'].values
test_texts = df.iloc[40000:]['review'].values
test_labels = df.iloc[40000:]['sentiment'].values

##### Tokenizing the Dataset

So far, we have obtained the texts and labels for the training, validation, and the test sets. Now, we are going to tokenize the texts into individual word tokens using the tokenizer implementation inherited from the pre-trained model class:

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
valid_encodings = tokenizer(list(valid_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

If you are interested in applying different types of tokenizers, feel free to explore the tokenizers package, which is also built and maintained by Hugging Face. However, inherited tokenizers maintain the consistency between the pre-trained model and the dataset, which saves us the extra effort of finding the specific tokenizer corresponding to the model. In other words, using an inherited tokenizer is the recommended appraoch if you want to fine-tune a pre-trained model.

Finally, let's pack everything into a class called IMDbDataset and create the corresponding data loaders. Such a self-defined dataset class lets us customize all the related features and functions for our custom movie review dataset in DataFrame format:

In [41]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx])
                for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
valid_dataset = IMDbDataset(valid_encodings, valid_labels)
test_dataset = IMDbDataset(test_encodings,  test_labels)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=16, shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=16, shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=16, shuffle=False
)

While the overall data loader setup should be familiar from previous chapters, one noteworthy detail is the item variable in the `__getitem__` method. The encodings we produced previously store a lot of information about the tokenized texts. Via the dictionary comprehension that we use to assign the dictionary to the item variable, we are only extacting the most relevant information. For instance, the resulting dictionary entries include input_ids (unique integers from the vocabulary corresponding to the tokens), labels (the class labels), and attention_mask. Here, the attenion_mask is a tensor with binary values (0s and 1s) that denotes which tokens the model should attend to. In particular, 0s correspond to tokens used for padding the sequence to equal lengths and are ignored by the model; the 1s correspond to the actual text tokens.

##### Loading and Fine-Tuning a Pre-Trained BERT Model

Having taken care of the data preparation, in this subsection, we will see how to load the pre-trained DistilBERT model and fine-tune it using the dataset we just created. The code for loading the pre-trained model is as follows:

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.to(DEVICE)
model.train()

optim = torch.optim.Adam(model.parameters(), lr=5e-5)

DistilBertForSequenceClassification specifies the downstream task we want to fine-tune the model on, which is sequence classification in this case. As mentioned before, 'distilbert-base-uncased' is a lightweight version of BERT uncased base model with manageable size and good performance. Note that 'uncased' means that the model does not distinguish between upper- and lower-case letters.

Now, it's time to train the model. We can break this up into two parts. First, we need to define an accuracy function to evaluate the model performance. Note that this accuracy function computes the conventional classification accuracy. Why is it so verbose? Here, we are loading the dataset batch by batch to work around RAM or GPU memory(VRAM) limitations when working with a large deep learning model:

In [43]:
def compute_accuracy(model, data_loader, device):
    with torch.no_grad():
        correct_pred, num_examples = 0, 0
        for batch_idx, batch in enumerate(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            predicted_labels = torch.argmax(logits, 1)
            num_examples += labels.size(0)
            correct_pred += (predicted_labels == labels).sum()
    return correct_pred.float()/num_examples * 100

In the `compute_accuracy` function, we load a given batch and then obtain the predicted labels from the outputs. While doing this, we keep track of the total number of exampes via num_examples. Similarly, we keep track of the number of correct predictions via the correct_pred variable. Finally, after we iterate over the complete dataset, we compute the accuracy as the proportion of correctly predicted labels.

Overall, via the `compute_accuracy` function, we can already get a glimpse at how we can use the transformer model to obtain the class label. That is, we feed the model the `input_ids` along with the `attention_mask` information that, here, denotes whether a token is an actual text token or a token for padding the sequences to equal length. The model call then returns the outputs, which is a transformer library-specific `SequenceClassifierOutput` object. From this object, we then obtain the logits that we convert into class labels via the `argmax` function as we have done in previous chapters.

Finally, let us get to the main part: the training (or rather, fine-tuning) loop. As we will notice, fine-tuning a model from the transformers library is very similar to training a model in pure PyTorch from scratch:

In [44]:
start_time = time.time()

for epoch in range(NUM_EPOCHS):

    model.train()

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optim.zero_grad()
        outputs = model(input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss, logits = outputs['loss'], outputs['logits']


        loss.backward()
        optim.step()

        if not batch_idx % 250:
            print(f'Epoch: {epoch+1:04d}/{NUM_EPOCHS:04d}'
                  f' | Batch '
                  f'{batch_idx:04d}/'
                  f'{len(train_loader):04d} | '
                  f'Loss: {loss:.4f}')

    model.eval()

    with torch.set_grad_enabled(False):
        print(f'Training accuracy: '
                f'{compute_accuracy(model, train_loader, DEVICE):.2f}%'
                f'\nValid accuracy: '
                f'{compute_accuracy(model, valid_loader, DEVICE):.2f}%')
        print(f'Time elapse: {(time.time() - start_time)/60:.2f} min')

print(f'Total Training Time: {(time.time() - start_time)/60.:.2f} min')
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 0001/0003 | Batch 0000/2188 | Loss: 0.7004
Epoch: 0001/0003 | Batch 0250/2188 | Loss: 0.3009
Epoch: 0001/0003 | Batch 0500/2188 | Loss: 0.4491
Epoch: 0001/0003 | Batch 0750/2188 | Loss: 0.0416
Epoch: 0001/0003 | Batch 1000/2188 | Loss: 0.0937
Epoch: 0001/0003 | Batch 1250/2188 | Loss: 0.0398
Epoch: 0001/0003 | Batch 1500/2188 | Loss: 0.2384
Epoch: 0001/0003 | Batch 1750/2188 | Loss: 0.0744
Epoch: 0001/0003 | Batch 2000/2188 | Loss: 0.1924
Training accuracy: 96.04%
Valid accuracy: 92.04%
Time elapse: 43.02 min
Epoch: 0002/0003 | Batch 0000/2188 | Loss: 0.0403
Epoch: 0002/0003 | Batch 0250/2188 | Loss: 0.0497
Epoch: 0002/0003 | Batch 0500/2188 | Loss: 0.0282
Epoch: 0002/0003 | Batch 0750/2188 | Loss: 0.0156
Epoch: 0002/0003 | Batch 1000/2188 | Loss: 0.1074
Epoch: 0002/0003 | Batch 1250/2188 | Loss: 0.2711
Epoch: 0002/0003 | Batch 1500/2188 | Loss: 0.0627
Epoch: 0002/0003 | Batch 1750/2188 | Loss: 0.2186
Epoch: 0002/0003 | Batch 2000/2188 | Loss: 0.3060
Training accuracy: 98.71%
Va

In this code, we iterate over multiple epochs. In each epoch we perform the following steps:
- Load the input into the device we are working on (GPU or CPU)
- Compute the model output and loss
- Adjust the weight parameters by backpropagating the loss
- Evaluate the model performance on both the training and validation set

Note that the training time may vary on different devices. After three epochs, accuracy on the test datset is around 92 percent, which is an improvement compared to the test accuracy that the RNN achieved in the previous chapter.

#### Fine-Tuning a Transformer More Conveniently Using the Trainer API

In the previous subsection, we implemented the training loop in PyTorch manually to illustrate that fine-tuning a transformer model is really not that much differernt from training an RNN or CNN model from scratch. However, note that the `transformers` library contains several nice features for additional convenience, like the Trainer API, which we will introduce in this subsection.

The Trainer API provided by Hugging Face is optimized for transformer models with a wide range of training options and various built-in features. When using the Trainer API, we can skip the effort of writing training loops on our own, and training or fine-tuning a transformer model is as simple as a function (or method) call. Let's see how this works in practice.

After loading the pre-trianed model via:

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.to(DEVICE)
model.train()

The training loop from the previous section can then be repalced by the following code:

In [47]:
optim = torch.optim.Adam(model.parameters(), lr=5e-5)
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    optimizers=(optim, None) # optim and learning rate scheduler
)

In the preceding code snippets, we first defined the training arguments, which are relatively self-explanatory settings regarding the input and output locations, number of epochs, and batch sizes We tried to keep the settings as simple as possible; however, there are many additional settings available.

We then passed these TrainingArguments settings to the Trainer class to instantiate a new trainer object. After initiating the trainer with the settings, the model to be fine-tuned, and the training and the evaluation sets, we can train the model by calling the `trainer.train()` method (we will use this method further shortly). That's it, using the Trainer API is as simple as shown in the previous code, and no further boilerplate code is required.

However, you may have noticed that the test dataset was not involved in these code snippets, and we haven't specified any evaluation metrics
 in this subsectino. This is because the Trainer API only shows the training loss and does not provide model evaluation along the training process by default. There are two ways to display the final model performance, which we will illustrate next.

The first method for evaluating the final model is to define an evaluation function as the `compute_metrics` argument for another Trainer instance. The `compute_metrics` function operates on the models' test predictions as logits (which is the default output of the model) and the test labels. To instantiate this function, we recommend installing Hugging Face's datasets library via pip install evaluate and use it as follows:

In [55]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  return metric.compute(predictions=predictions, references=labels)

The updated Trainer instantiation (now including `compute_metrics`) is then as follows:

In [56]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    optimizers=(optim, None)
)

Now, let's train the model (again, note that the code is not fully deterministic, which is why you might get slightly different results):

In [ ]:
start_time = time.time()
trainer.train()

In [53]:
print(f'Total Training Time: '
      f'{(time.time() - start_time)/60:.2f} min')

Total Training Time: 94.13 min


After the training has completed, which can take up to an hour depending on your GPU, we can call trainer.evaluate() to obtain the model performance on the test set:

In [57]:
print(trainer.evaluate())

Training Loss,Validation Loss,Step,Accuracy
No log,0.288034,0,0.937300


{'eval_loss': 0.288034051656723, 'eval_accuracy': 0.9373}


As we can see, the evaluation accuracy is around 94 percent, similar to our own previously used PyTorch training loop. (Note that we have skipped the training step, because the model is already fine-tuned after the previous trainer.train() call.) There is a small discrepency between our manual training approach and using the Trainer class, because the Trainer class uses some different and some additional settings.

The second method we could employ to compute the final test set accuracy is re-using our `compute_accuracy` function that we defined in the previous section. We can directly evaluate the performance of the fine-tuned model on the test dataset by running the following code:

In [58]:
model.eval()
model.to(DEVICE)

print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Test accuracy: 93.73%


In fact, if you want to check the model's performance regularly during training, you can require the trainer to print the model evaluation after each epoch by defining the training arguments as follows:

In [60]:
from transformers import TrainingArguments

training_args = TrainingArguments("test_trainer",
                                  eval_strategy="epoch")

However, if you are planning to change or optimize hyperparameters and repeat the fine-tuning procedure several times, we recommend using the validation set for this purpose, in order to keep the test set independent. We can achieve this by instantiating the Trainer using the valid_dataset:

In [61]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)